# Understanding unfold method in PyTorch

In [1]:
import torch

In [2]:
t1 = torch.arange(8)

In [3]:
t1

tensor([0, 1, 2, 3, 4, 5, 6, 7])

In [4]:
t1.unfold(0, 2, 2)

tensor([[0, 1],
        [2, 3],
        [4, 5],
        [6, 7]])

### What is `torch.Tensor.unfold`

`unfold` is a tensor method that extracts sliding local blocks from a specified dimension without copying all data into a new tensor. It is useful for creating windowed views of input data, such as image patches or overlapping segments in sequences.

### Parameters

- `dimension`: the axis along which to extract windows.
- `size`: the length of each slice or window.
- `step`: the stride between the starts of consecutive windows.

For example, `tensor.unfold(2, 16, 16)` means:
- take windows along dimension 2,
- each window has size 16,
- move 16 steps for the next window.

### Output shape

If the original size along the selected dimension is `L`, and you unfold with `size=S` and `step=K`, the number of windows is:
- `n = floor((L - S) / K) + 1`

The result tensor has an extra dimension appended at the end for the window contents. So a tensor of shape `[B, C, H, W]` unfolded on height and width becomes:
- `[B, C, H_windows, W_windows, S, S]`

This means you now have a grid of patches, where each patch is a slice of length `S` along the unfolded dimension.

### How it works

`unfold` does not reshape the tensor in the usual flattening sense. Instead, it creates a view where one dimension indexes the window positions and another dimension contains the window values. That is why it is efficient for patch extraction and local operations.

### Example usage in patch extraction

For image patching, apply `unfold` twice:
- once on the height dimension,
- once on the width dimension.

This converts a tensor of shape `[batch, channels, height, width]` into a tensor of patches:
- `[batch, channels, num_patches_h, num_patches_w, patch_size, patch_size]`

Then you can reshape or permute to get a list of flattened patches if needed.

### Common use cases

- extracting image patches for Vision Transformers
- creating sliding windows over time series data
- building local receptive fields without explicit loops
- reducing operations to patch-based processing in CNN-like pipelines

### Formula

(number_of_windows)

= floor((length-size)/step)+1

In [12]:
t1.unfold(0, 3, 2)

tensor([[0, 1, 2],
        [2, 3, 4],
        [4, 5, 6]])

### Two Dimensional Example

In [16]:
x2 = torch.arange(16)
x2

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15])

In [23]:
x2 = x2.reshape(4, 4)
x2

tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11],
        [12, 13, 14, 15]])

In [32]:
x2.unfold(0, 3, 2)

tensor([[[ 0,  4,  8],
         [ 1,  5,  9],
         [ 2,  6, 10],
         [ 3,  7, 11]]])

In [33]:
x2.unfold(1, 3, 2)

tensor([[[ 0,  1,  2]],

        [[ 4,  5,  6]],

        [[ 8,  9, 10]],

        [[12, 13, 14]]])

### Image patches

In [34]:
x2.unfold(0,2,1).unfold(1,2,1)

tensor([[[[ 0,  1],
          [ 4,  5]],

         [[ 1,  2],
          [ 5,  6]],

         [[ 2,  3],
          [ 6,  7]]],


        [[[ 4,  5],
          [ 8,  9]],

         [[ 5,  6],
          [ 9, 10]],

         [[ 6,  7],
          [10, 11]]],


        [[[ 8,  9],
          [12, 13]],

         [[ 9, 10],
          [13, 14]],

         [[10, 11],
          [14, 15]]]])